# Papa CTD Cast Comparison

The purpose of this notebook is to analyze the papa cast data and determine if we can do a single cast

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("../data/923122_v2_ooi_station_papa_ctd_and_water_sampling_data.csv")

In [3]:
data.head()

,Cruise,Station,Target_Asset,Start_Latitude,Start_Longitude,Start_Time,Cast,Cast_Flag,Bottom_Depth_at_Start_Position,CTD_File,...,Discrete_pH_Replicate_Flag,Calculated_Alkalinity,Calculated_DIC,Calculated_pCO2,Calculated_pH,Calculated_CO2aq,Calculated_Bicarb,Calculated_CO3,Calculated_Omega_C,Calculated_Omega_A
0,MV1309,001,CTDMO cal/val & Release Testing,49.532333,-135.051,2013-07-17T19:31:24.000Z,001,*0000000000000100,3635.0,MV1309_CTD01.hex,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MV1309,001,CTDMO cal/val & Release Testing,49.532333,-135.051,2013-07-17T19:31:24.000Z,001,*0000000000000100,3635.0,MV1309_CTD01.hex,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MV1309,001,CTDMO cal/val & Release Testing,49.532333,-135.051,2013-07-17T19:31:24.000Z,001,*0000000000000100,3635.0,MV1309_CTD01.hex,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MV1309,001,CTDMO cal/val & Release Testing,49.532333,-135.051,2013-07-17T19:31:24.000Z,001,*0000000000000100,3635.0,MV1309_CTD01.hex,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MV1309,001,CTDMO cal/val & Release Testing,49.532333,-135.051,2013-07-17T19:31:24.000Z,001,*0000000000000100,3635.0,MV1309_CTD01.hex,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## T-S Diagram

Compare water-mass properties across casts using a temperature-salinity diagram. `CTD_Salinity_1` and `CTD_Salinity_2` each go bad on different casts in different ways (e.g. RB1605 cast 003 has bad `_1` spikes below ~1000 dbar; SR1811 cast 003 has a sustained bad `_2` drift down to ~25 PSU near 50-325 dbar) — confirmed against the sparse `Discrete_Salinity` bottle samples. Neither sensor is reliable on its own, so for each cast we average the two sensors where they agree, then resolve disagreements against a smooth per-cast reference profile built from the agreeing points.

In [ ]:
ts = data.dropna(subset=["Cast"]).copy()

# Keep only physically plausible readings (Station Papa salinity ~25-36 PSU);
# this alone drops the gross sensor failures (e.g. values of ~72-112 or 1999).
s1 = ts["CTD_Salinity_1"].where(ts["CTD_Salinity_1"].between(25, 36))
s2 = ts["CTD_Salinity_2"].where(ts["CTD_Salinity_2"].between(25, 36))
agree = (s1 - s2).abs() < 0.3


def resolve_salinity(cast):
    """Average sensors where they agree; elsewhere pick whichever sensor is
    closer to a pressure-interpolated reference built from the agreeing
    points in this same cast."""
    idx = cast.index
    pressure = cast["CTD_Pressure"]
    a, cast_s1, cast_s2 = agree.loc[idx], s1.loc[idx], s2.loc[idx]
    avg = (cast_s1 + cast_s2) / 2

    salinity = pd.Series(np.nan, index=idx)
    salinity[a] = avg[a]

    if a.sum() >= 2:
        order = np.argsort(pressure[a].to_numpy())
        reference = pd.Series(
            np.interp(pressure, pressure[a].to_numpy()[order], avg[a].to_numpy()[order]),
            index=idx,
        )
        disagree = ~a
        pick_s1 = disagree & cast_s1.notna() & ((cast_s1 - reference).abs() <= (cast_s2 - reference).abs())
        pick_s2 = disagree & cast_s2.notna() & ~pick_s1
        salinity[pick_s1] = cast_s1[pick_s1]
        salinity[pick_s2] = cast_s2[pick_s2]

    # No agreeing pair to build a reference from: fall back to whichever sensor is valid.
    remaining = salinity.isna()
    salinity[remaining] = cast_s1[remaining].fillna(cast_s2[remaining])
    return salinity


ts["Salinity"] = ts.groupby(["Cruise", "Cast"], group_keys=False).apply(resolve_salinity, include_groups=False).reindex(ts.index)
ts["Temperature"] = ts["CTD_Temperature_1"]

# CTD_Oxygen is unusable on a couple of casts: SR1811 010/10b below ~900 dbar
# carries the -99 "no data" sentinel in CTD_Oxygen_Saturation alongside
# nonsense oxygen values (up to 32), and SKQ202308S casts 008-010 have a
# miscalibrated sensor reading negative throughout (confirmed against the
# Discrete_Oxygen bottle samples, which show a normal ~0-7 mL/L profile for
# those same casts). Drop both physically implausible negatives and sentinel
# rows.
ts["Oxygen"] = ts["CTD_Oxygen"].where((ts["CTD_Oxygen"] >= 0) & (ts["CTD_Oxygen_Saturation"] > -50))

ts = ts.dropna(subset=["Salinity", "Temperature"])
print(f"{len(ts)} of {len(data)} rows retained after QC")
ts[["Cruise", "Cast", "CTD_Pressure", "Temperature", "Salinity", "Oxygen"]].describe()

In [5]:
def sigma_t(temp, sal):
    """Density anomaly (sigma-t, kg/m^3) at atmospheric pressure.

    UNESCO 1980 / Millero & Poisson (1981) equation of state, evaluated with
    in-situ temperature as a stand-in for potential temperature (fine at the
    near-surface pressures relevant to a T-S diagram background grid).
    """
    t = np.asarray(temp, dtype=float)
    s = np.asarray(sal, dtype=float)

    rho_w = (
        999.842594 + 6.793952e-2 * t - 9.095290e-3 * t**2
        + 1.001685e-4 * t**3 - 1.120083e-6 * t**4 + 6.536332e-9 * t**5
    )
    A = 8.24493e-1 - 4.0899e-3 * t + 7.6438e-5 * t**2 - 8.2467e-7 * t**3 + 5.3875e-9 * t**4
    B = -5.72466e-3 + 1.0227e-4 * t - 1.6546e-6 * t**2
    C = 4.8314e-4

    rho = rho_w + A * s + B * s**1.5 + C * s**2
    return rho - 1000

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential blue ramp (light -> dark) to encode cruise chronology (cruise
# codes are already in time order at Station Papa, 2013-2025).
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95", "#0d366b"]
CRUISE_CMAP = LinearSegmentedColormap.from_list("cruise_time", BLUE_RAMP)


def plot_ts_diagram(df, title="T-S Diagram", figsize=(8, 7), ax=None):
    """Plot a temperature-salinity diagram with sigma-t density contours,
    colored by cruise (light = earliest, dark = latest; cruise codes in this
    dataset are already chronological). `df` must have Salinity, Temperature,
    and Cruise columns, e.g. `ts`, `flma`, `flmb`, or `hypm`."""
    cruises = df["Cruise"].unique().tolist()
    cruise_color = {
        cruise: CRUISE_CMAP(i / max(len(cruises) - 1, 1))
        for i, cruise in enumerate(cruises)
    }

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    s_grid = np.linspace(df["Salinity"].min() - 0.2, df["Salinity"].max() + 0.2, 200)
    t_grid = np.linspace(df["Temperature"].min() - 0.5, df["Temperature"].max() + 0.5, 200)
    S_grid, T_grid = np.meshgrid(s_grid, t_grid)
    density = sigma_t(T_grid, S_grid)
    contours = ax.contour(S_grid, T_grid, density, colors="#c3c2b7", linewidths=0.8, levels=10)
    ax.clabel(contours, inline=True, fontsize=8, fmt="%.1f", colors="#898781")

    for cruise in cruises:
        sub = df[df["Cruise"] == cruise]
        ax.scatter(
            sub["Salinity"], sub["Temperature"],
            s=14, color=cruise_color[cruise], label=cruise,
            edgecolors="none", alpha=0.85,
        )

    ax.set_xlabel("Salinity (PSU)")
    ax.set_ylabel("Temperature (°C)")
    ax.set_title(f"{title}\n(background contours = $\\sigma_t$ density, kg/m$^3$)")
    ax.legend(title="Cruise", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=1.5)
    fig.tight_layout()
    return fig, ax


fig, ax = plot_ts_diagram(ts, title="Station Papa CTD Casts: T-S Diagram")
plt.show()

## Split Casts by Target Asset

`Target_Asset` is free text (82 distinct values) rather than a clean category, so we match it with case-insensitive substrings: `FLMA`, `FLMB` (also matching the combined `FLMA/B` spelling, which doesn't contain the literal substring `FLMB`), and `HYPM`. A handful of entries name more than one target (e.g. `"FLMA/B cal/val;near gliders,HYPM"`), so a cast can land in more than one subset — that's intentional, not a bug.

In [ ]:
def target_mask(target_asset, *patterns):
    return target_asset.str.contains("|".join(patterns), case=False, na=False, regex=True)


flma_mask = target_mask(ts["Target_Asset"], "FLMA")
flmb_mask = target_mask(ts["Target_Asset"], "FLMB", r"FLMA\s*/\s*B")
hypm_mask = target_mask(ts["Target_Asset"], "HYPM")

flma = ts[flma_mask].copy()
flmb = ts[flmb_mask].copy()
hypm = ts[hypm_mask].copy()

for name, subset, mask in [("FLMA", flma, flma_mask), ("FLMB", flmb, flmb_mask), ("HYPM", hypm, hypm_mask)]:
    n_casts = subset[["Cruise", "Cast"]].drop_duplicates().shape[0]
    print(f"{name}: {mask.sum()} rows, {n_casts} unique casts")

unmatched = ts[~(flma_mask | flmb_mask | hypm_mask)]
print(f"\nUnmatched: {len(unmatched)} rows")
unmatched["Target_Asset"].value_counts()

In [ ]:
for name, subset in [("FLMA", flma), ("FLMB", flmb), ("HYPM", hypm)]:
    plot_ts_diagram(subset, title=f"Station Papa CTD Casts near {name}: T-S Diagram")
    plt.show()

## Depth Profiles: Temperature, Salinity, Dissolved Oxygen

Same idea as the T-S diagrams, but as vertical profiles against pressure so each variable's structure with depth is visible directly, again colored by cruise (light = earliest, dark = latest).

In [ ]:
PROFILE_VARS = [
    ("Temperature", "Temperature (°C)"),
    ("Salinity", "Salinity (PSU)"),
    ("Oxygen", "Dissolved Oxygen (mL/L)"),
]


def plot_profiles(df, title="Profile Comparison", figsize=(13, 7)):
    """Plot Temperature, Salinity, and Dissolved Oxygen vs pressure side by
    side, colored by cruise (light = earliest, dark = latest; cruise codes
    in this dataset are already chronological). `df` must have Temperature,
    Salinity, Oxygen, CTD_Pressure, and Cruise columns, e.g. `ts`, `flma`,
    `flmb`, or `hypm`."""
    cruises = df["Cruise"].unique().tolist()
    cruise_color = {
        cruise: CRUISE_CMAP(i / max(len(cruises) - 1, 1))
        for i, cruise in enumerate(cruises)
    }

    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, (col, xlabel) in zip(axes, PROFILE_VARS):
        for cruise in cruises:
            sub = df.loc[df["Cruise"] == cruise, [col, "CTD_Pressure"]].dropna()
            ax.scatter(
                sub[col], sub["CTD_Pressure"],
                s=10, color=cruise_color[cruise], label=cruise,
                edgecolors="none", alpha=0.85,
            )
        ax.set_xlabel(xlabel)
        ax.grid(color="#e1e0d9", linewidth=0.6)

    axes[0].invert_yaxis()
    axes[0].set_ylabel("Pressure (dbar)")
    axes[-1].legend(title="Cruise", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=1.5)
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


for name, subset in [("FLMA", flma), ("FLMB", flmb), ("HYPM", hypm)]:
    plot_profiles(subset, title=f"Station Papa CTD Casts near {name}: Depth Profiles")
    plt.show()

## Within-Cruise Cast-to-Cast Differences

A cruise doesn't always visit a given target exactly twice (e.g. MV1404 has 4 different casts tagged FLMA — a cal/val, its repeat, a gliders cast, and a recovery cast), so for every cruise we take **every pairwise combination** of casts at the same target and difference them. Since the two casts in a pair rarely land on identical pressure levels, each pair is interpolated onto a shared pressure grid (10 dbar steps, restricted to the pressure range both casts actually cover) before differencing. The pair is ordered chronologically by `Start_Time`, so `Difference = earlier cast − later cast` throughout.

In [ ]:
import itertools


def pairwise_cast_differences(df, var, pressure_step=10, as_percent=False):
    """For each cruise in `df`, interpolate every pairwise combination of
    casts onto a shared pressure grid and difference them for `var`
    (earlier cast minus later cast, by Start_Time). With as_percent=True,
    the difference is expressed as a percentage of the pair's mean value at
    that pressure level instead of in the variable's own units. Returns a
    long-format DataFrame with one row per (Cruise, Cast_A, Cast_B, Pressure)."""
    rows = []
    for cruise, group in df.groupby("Cruise"):
        casts = (
            group[["Cast", "Start_Time"]].drop_duplicates()
            .sort_values("Start_Time")["Cast"].tolist()
        )
        profiles = {}
        for cast in casts:
            sub = group.loc[group["Cast"] == cast, ["CTD_Pressure", var]].dropna().sort_values("CTD_Pressure")
            if len(sub) >= 2:
                profiles[cast] = (sub["CTD_Pressure"].to_numpy(), sub[var].to_numpy())

        for cast_a, cast_b in itertools.combinations([c for c in casts if c in profiles], 2):
            pa, va = profiles[cast_a]
            pb, vb = profiles[cast_b]
            lo, hi = max(pa.min(), pb.min()), min(pa.max(), pb.max())
            if hi <= lo:
                continue
            grid = np.arange(lo, hi, pressure_step)
            interp_a = np.interp(grid, pa, va)
            interp_b = np.interp(grid, pb, vb)
            diff = interp_a - interp_b
            if as_percent:
                mean_val = (interp_a + interp_b) / 2
                diff = np.where(mean_val != 0, diff / mean_val * 100, np.nan)
            rows.append(pd.DataFrame({
                "Cruise": cruise, "Cast_A": cast_a, "Cast_B": cast_b,
                "Pressure": grid, "Difference": diff,
            }))

    columns = ["Cruise", "Cast_A", "Cast_B", "Pressure", "Difference"]
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=columns)


def plot_pairwise_differences(df, title="Within-Cruise Cast Differences", figsize=(13, 7), as_percent=False):
    """For Temperature, Salinity, and Oxygen, plot every within-cruise
    pairwise cast-vs-cast difference against pressure, colored by cruise
    (light = earliest, dark = latest). `df` must be a target subset with
    Temperature/Salinity/Oxygen/CTD_Pressure/Cruise/Cast/Start_Time columns,
    e.g. `flma`, `flmb`, or `hypm`. With as_percent=True, differences are
    plotted as a percentage of each pair's mean value instead of raw units."""
    cruises = df["Cruise"].unique().tolist()
    cruise_color = {
        cruise: CRUISE_CMAP(i / max(len(cruises) - 1, 1))
        for i, cruise in enumerate(cruises)
    }

    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, (col, xlabel) in zip(axes, PROFILE_VARS):
        diffs = pairwise_cast_differences(df, col, as_percent=as_percent)
        for cruise in cruises:
            sub = diffs[diffs["Cruise"] == cruise]
            for _, pair in sub.groupby(["Cast_A", "Cast_B"]):
                ax.plot(
                    pair["Difference"], pair["Pressure"],
                    color=cruise_color[cruise], linewidth=1.2, alpha=0.85, label=cruise,
                )
        ax.axvline(0, color="#c3c2b7", linewidth=0.8)
        ax.set_xlabel(f"% Δ {col}" if as_percent else f"Δ {xlabel}")
        ax.grid(color="#e1e0d9", linewidth=0.6)

    axes[0].invert_yaxis()
    axes[0].set_ylabel("Pressure (dbar)")
    handles, labels = axes[-1].get_legend_handles_labels()
    by_cruise = dict(zip(labels, handles))  # one legend entry per cruise, not per pair
    axes[-1].legend(by_cruise.values(), by_cruise.keys(), title="Cruise", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


for name, subset in [("FLMA", flma), ("FLMB", flmb), ("HYPM", hypm)]:
    plot_pairwise_differences(subset, title=f"{name}: Within-Cruise Cast-to-Cast Differences")
    plt.show()

### As Percent Differences

Same pairing and interpolation, but each difference is expressed as a percentage of the pair's mean value at that pressure level (`(A - B) / mean(A, B) * 100`) instead of raw units — set `as_percent=True` on either function. Two caveats: Oxygen differences blow up wherever both casts are near-zero (the OMZ core around 500-1000 dbar), since a small denominator amplifies noise into large percentages; and percent Temperature is relative to Celsius, which isn't a ratio scale (0 °C isn't "no temperature"), so treat it as a rough normalization rather than a physically meaningful percentage — the absolute °C version above is more defensible for temperature specifically.

In [ ]:
for name, subset in [("FLMA", flma), ("FLMB", flmb), ("HYPM", hypm)]:
    plot_pairwise_differences(subset, title=f"{name}: Within-Cruise Cast-to-Cast Percent Differences", as_percent=True)
    plt.show()

## Cross-Target Differences: Mean Profile per Target, per Cruise

Now compare *across* targets instead of within one: for each cruise, build one mean profile per target (FLMA/FLMB/HYPM) by interpolating that cruise's casts at that target onto a shared pressure grid — restricted to each cast's own pressure range, so no cast is extrapolated past where it actually sampled — and averaging. Then, for each pair of targets, difference their mean profiles for every cruise that visited both (`Difference = Target_A mean − Target_B mean`). This shows whether a single target's cast could stand in for the others within the same cruise, as opposed to the within-target repeatability checked above.

In [ ]:
TARGET_DFS = {"FLMA": flma, "FLMB": flmb, "HYPM": hypm}


def target_mean_profiles(df, var, pressure_step=10):
    """Per cruise in `df`, average all of that cruise's casts for `var` onto
    a shared pressure grid, without extrapolating any cast past its own
    pressure range. Returns {cruise: (pressure_array, mean_value_array)}."""
    result = {}
    for cruise, group in df.groupby("Cruise"):
        cast_profiles = []
        for _, csub in group.groupby("Cast"):
            sub = csub[["CTD_Pressure", var]].dropna().sort_values("CTD_Pressure")
            if len(sub) >= 2:
                cast_profiles.append((sub["CTD_Pressure"].to_numpy(), sub[var].to_numpy()))
        if not cast_profiles:
            continue

        lo = min(p.min() for p, _ in cast_profiles)
        hi = max(p.max() for p, _ in cast_profiles)
        grid = np.arange(lo, hi, pressure_step)

        stack = []
        for p, v in cast_profiles:
            vals = np.interp(grid, p, v)
            vals[(grid < p.min()) | (grid > p.max())] = np.nan
            stack.append(vals)
        # Grid points outside every cast's coverage (a gap between casts'
        # ranges) are all-NaN; nanmean warns there, which we drop below anyway.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean_vals = np.nanmean(np.vstack(stack), axis=0)

        valid = ~np.isnan(mean_vals)
        result[cruise] = (grid[valid], mean_vals[valid])
    return result


def target_pair_differences(target_a, target_b, var, pressure_step=10, as_percent=False):
    """Difference the mean target_a and target_b profiles for `var`, for
    every cruise that has both (Difference = target_a mean - target_b mean)."""
    means_a = target_mean_profiles(TARGET_DFS[target_a], var, pressure_step)
    means_b = target_mean_profiles(TARGET_DFS[target_b], var, pressure_step)

    rows = []
    for cruise in sorted(set(means_a) & set(means_b)):
        pa, va = means_a[cruise]
        pb, vb = means_b[cruise]
        lo, hi = max(pa.min(), pb.min()), min(pa.max(), pb.max())
        if hi <= lo or len(pa) < 2 or len(pb) < 2:
            continue
        grid = np.arange(lo, hi, pressure_step)
        interp_a = np.interp(grid, pa, va)
        interp_b = np.interp(grid, pb, vb)
        diff = interp_a - interp_b
        if as_percent:
            mean_val = (interp_a + interp_b) / 2
            diff = np.where(mean_val != 0, diff / mean_val * 100, np.nan)
        rows.append(pd.DataFrame({"Cruise": cruise, "Pressure": grid, "Difference": diff}))

    columns = ["Cruise", "Pressure", "Difference"]
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=columns)


def plot_target_pair_differences(target_a, target_b, figsize=(13, 7), as_percent=False):
    """Plot the target_a-mean minus target_b-mean difference for
    Temperature, Salinity, and Oxygen against pressure, one line per cruise
    that visited both targets, colored by cruise (light = earliest, dark =
    latest)."""
    all_cruises = ts["Cruise"].unique().tolist()  # fixed chronological order/coloring
    cruise_color = {
        cruise: CRUISE_CMAP(i / max(len(all_cruises) - 1, 1))
        for i, cruise in enumerate(all_cruises)
    }

    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, (col, xlabel) in zip(axes, PROFILE_VARS):
        diffs = target_pair_differences(target_a, target_b, col, as_percent=as_percent)
        for cruise, cast_df in diffs.groupby("Cruise"):
            ax.plot(
                cast_df["Difference"], cast_df["Pressure"],
                color=cruise_color[cruise], linewidth=1.2, alpha=0.85, label=cruise,
            )
        ax.axvline(0, color="#c3c2b7", linewidth=0.8)
        ax.set_xlabel(f"% Δ {col}" if as_percent else f"Δ {xlabel}")
        ax.grid(color="#e1e0d9", linewidth=0.6)

    axes[0].invert_yaxis()
    axes[0].set_ylabel("Pressure (dbar)")
    handles, labels = axes[-1].get_legend_handles_labels()
    by_cruise = dict(zip(labels, handles))
    axes[-1].legend(by_cruise.values(), by_cruise.keys(), title="Cruise", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.suptitle(f"{target_a} mean − {target_b} mean, by cruise")
    fig.tight_layout()
    return fig, axes


for target_a, target_b in itertools.combinations(TARGET_DFS, 2):
    plot_target_pair_differences(target_a, target_b)
    plt.show()